In [1]:
import os
import numpy as np
import pandas as pd
import scanpy as sc
import anndata 
import seaborn as sns
from scipy.stats import zscore
import matplotlib.pyplot as plt
import collections
from natsort import natsorted

from scipy import stats
from scipy import sparse
from sklearn.decomposition import PCA
from umap import UMAP
from statsmodels.stats.multitest import multipletests

from matplotlib.colors import LinearSegmentedColormap

from scroutines.config_plots import *
from scroutines import powerplots # .config_plots import *
from scroutines import pnmf
from scroutines import basicu
from scroutines.gene_modules import GeneModules  

import sys
sys.path.insert(0, '/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/myvisctx/analysis_atac/')
import atac_utils

In [2]:
ddir = '/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/data/v1_multiome/from_sanjana/' 
!ls $ddir

L2_3_direct_e_regulon_metadata.csv    L5IT_direct_e_regulon_metadata.csv
L2_3_extended_e_regulon_metadata.csv  L5IT_extended_e_regulon_metadata.csv
L4_direct_e_regulon_metadata.csv      L6IT_direct_e_regulon_metadata.csv
L4_extended_e_regulon_metadata.csv    L6IT_extended_e_regulon_metadata.csv


## In take

In [3]:
f = ddir+'L2_3_extended_e_regulon_metadata.csv'
# scenic metadata
df_scenic = pd.read_csv(f, index_col=0)
df_scenic['signs'] = df_scenic['eRegulon_name'].apply(lambda x: x[-3]+x[-1])
df_scenic['Consensus_name'] = df_scenic['TF']+df_scenic['signs']
df_scenic = df_scenic[df_scenic['signs'].isin(["++", "-+"])]
df_scenic

,Region,Gene,importance_R2G,rho_R2G,importance_x_rho,importance_x_abs_rho,TF,is_extended,eRegulon_name,Gene_signature_name,Region_signature_name,importance_TF2G,regulation,rho_TF2G,triplet_rank,signs,Consensus_name
0,chr4:98321309-98321809,Patj,0.045634,0.169463,0.007733,0.007733,Aff4,True,Aff4_extended_+/+,Aff4_extended_+/+_(25g),Aff4_extended_+/+_(33r),0.933003,1,0.078527,17289,++,Aff4++
1,chr19:10290149-10290649,Dagla,0.048683,0.213159,0.010377,0.010377,Aff4,True,Aff4_extended_+/+,Aff4_extended_+/+_(25g),Aff4_extended_+/+_(33r),0.421422,1,0.081390,32578,++,Aff4++
2,chr11:53058505-53059005,Fstl4,0.033983,0.353861,0.012025,0.012025,Aff4,True,Aff4_extended_+/+,Aff4_extended_+/+_(25g),Aff4_extended_+/+_(33r),0.344324,1,0.117498,31034,++,Aff4++
3,chr19:10397699-10398199,Dagla,0.028603,0.199401,0.005703,0.005703,Aff4,True,Aff4_extended_+/+,Aff4_extended_+/+_(25g),Aff4_extended_+/+_(33r),0.421422,1,0.081390,36931,++,Aff4++
4,chr11:77865039-77865539,Myo18a,0.005209,0.097288,0.000507,0.000507,Aff4,True,Aff4_extended_+/+,Aff4_extended_+/+_(25g),Aff4_extended_+/+_(33r),0.469791,1,0.068899,27790,++,Aff4++
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41921,chr9:61899484-61899984,Rplp1,0.003120,0.217316,0.000678,0.000678,Zfp654,True,Zfp654_extended_-/+,Zfp654_extended_-/+_(10g),Zfp654_extended_-/+_(12r),0.103970,-1,-0.065798,41791,-+,Zfp654-+
41922,chr11:85841604-85842104,Brip1,0.013836,0.213501,0.002954,0.002954,Zfp654,True,Zfp654_extended_-/+,Zfp654_extended_-/+_(10g),Zfp654_extended_-/+_(12r),0.039431,-1,-0.059304,45590,-+,Zfp654-+
41923,chr9:61878788-61879288,Rplp1,0.034517,0.124719,0.004305,0.004305,Zfp654,True,Zfp654_extended_-/+,Zfp654_extended_-/+_(10g),Zfp654_extended_-/+_(12r),0.103970,-1,-0.065798,36330,-+,Zfp654-+
41924,chr7:44262665-44263165,Emc10,0.046827,0.110950,0.005195,0.005195,Zfp654,True,Zfp654_extended_-/+,Zfp654_extended_-/+_(10g),Zfp654_extended_-/+_(12r),0.062461,-1,-0.052993,12946,-+,Zfp654-+


In [4]:
df_reg = df_scenic.groupby(['TF', 'signs', 'Consensus_name',]).first()[['Region_signature_name', 'Gene_signature_name', ]].sort_values('TF')

scenic_regions = np.sort(df_scenic['Region'].unique()) # .shape
scenic_genes = np.sort(df_scenic['Gene'].unique()) # .shape
scenic_tfs = np.sort(df_scenic['TF'].unique())

num_reg    = len(df_reg)
num_tf     = len(scenic_tfs)
num_gene = len(scenic_genes)
num_region = len(scenic_regions)
print(num_reg, num_tf, num_gene, num_region)
df_reg

128 95 3945 14759


Region_signature_name  \
TF     signs Consensus_name                              
Aff4   ++    Aff4++            Aff4_extended_+/+_(33r)   
Arnt2  ++    Arnt2++          Arnt2_extended_+/+_(16r)   
Atf6   ++    Atf6++           Atf6_extended_+/+_(256r)   
       -+    Atf6-+            Atf6_extended_-/+_(18r)   
Bach2  ++    Bach2++          Bach2_extended_+/+_(36r)   
...                                                ...   
Zfp148 -+    Zfp148-+        Zfp148_extended_-/+_(30r)   
Zfp57  ++    Zfp57++          Zfp57_extended_+/+_(67r)   
       -+    Zfp57-+         Zfp57_extended_-/+_(121r)   
Zfp654 ++    Zfp654++        Zfp654_extended_+/+_(85r)   
       -+    Zfp654-+        Zfp654_extended_-/+_(12r)   

                                   Gene_signature_name  
TF     signs Consensus_name                             
Aff4   ++    Aff4++            Aff4_extended_+/+_(25g)  
Arnt2  ++    Arnt2++          Arnt2_extended_+/+_(16g)  
Atf6   ++    Atf6++           Atf6_extended_+/+_(203g)  
       -+    Atf6-+            Atf6_extended_-/+_(16g)  
Bach2  ++    Bach2++          Bach2_extended_+/+_(30g)  
...                                                ...  
Zfp148 -+    Zfp148-+        Zfp148_extended_-/+_(14g)  
Zfp57  ++    Zfp57++          Zfp57_extended_+/+_(58g)  
       -+    Zfp57-+          Zfp57_extended_-/+_(92g)  
Zfp654 ++    Zfp654++        Zfp654_extended_+/+_(66g)  
       -+    Zfp654-+        Zfp654_extended_-/+_(10g)  

[128 rows x 2 columns]

In [5]:
df_reg.loc['Meis2']

,,Region_signature_name,Gene_signature_name
signs,Consensus_name,,
++,Meis2++,Meis2_extended_+/+_(25r),Meis2_extended_+/+_(23g)


## In take 2

In [6]:
f = ddir+'L6IT_extended_e_regulon_metadata.csv'
# scenic metadata
df_scenic = pd.read_csv(f, index_col=0)
df_scenic['signs'] = df_scenic['eRegulon_name'].apply(lambda x: x[-3]+x[-1])
df_scenic['Consensus_name'] = df_scenic['TF']+df_scenic['signs']
df_scenic = df_scenic[df_scenic['signs'].isin(["++", "-+"])]
df_scenic

,Region,Gene,importance_R2G,rho_R2G,importance_x_rho,importance_x_abs_rho,TF,is_extended,eRegulon_name,Gene_signature_name,Region_signature_name,importance_TF2G,regulation,rho_TF2G,triplet_rank,signs,Consensus_name
0,chr18:4590581-4591081,Jcad,0.027395,0.334118,0.009153,0.009153,Arnt2,True,Arnt2_extended_+/+,Arnt2_extended_+/+_(16g),Arnt2_extended_+/+_(16r),0.584741,1,0.180278,13783,++,Arnt2++
1,chr13:63642602-63643102,Ptch1,0.070911,0.230684,0.016358,0.016358,Arnt2,True,Arnt2_extended_+/+,Arnt2_extended_+/+_(16g),Arnt2_extended_+/+_(16r),0.971550,1,0.122779,3722,++,Arnt2++
2,chr11:59808943-59809443,Nt5m,0.020012,0.131317,0.002628,0.002628,Arnt2,True,Arnt2_extended_+/+,Arnt2_extended_+/+_(16g),Arnt2_extended_+/+_(16r),1.321346,1,0.104384,2924,++,Arnt2++
3,chr13:97254016-97254516,Gfm2,0.006895,0.084715,0.000584,0.000584,Arnt2,True,Arnt2_extended_+/+,Arnt2_extended_+/+_(16g),Arnt2_extended_+/+_(16r),0.666900,1,0.103536,14821,++,Arnt2++
4,chr6:108660129-108660629,Arl8b,0.037479,0.067248,0.002520,0.002520,Arnt2,True,Arnt2_extended_+/+,Arnt2_extended_+/+_(16g),Arnt2_extended_+/+_(16r),1.090609,1,0.093177,3124,++,Arnt2++
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14264,chr4:126103101-126103601,Thrap3,0.027615,0.067861,0.001874,0.001874,Zfp57,True,Zfp57_extended_-/+,Zfp57_extended_-/+_(12g),Zfp57_extended_-/+_(12r),1.414784,-1,-0.068256,3429,-+,Zfp57-+
14265,chr17:47511070-47511570,Trerf1,0.025164,0.170645,0.004294,0.004294,Zfp57,True,Zfp57_extended_-/+,Zfp57_extended_-/+_(12g),Zfp57_extended_-/+_(12r),0.627162,-1,-0.104713,13379,-+,Zfp57-+
14266,chr2:181119936-181120436,Dnajc5,0.045163,0.153791,0.006946,0.006946,Zfp57,True,Zfp57_extended_-/+,Zfp57_extended_-/+_(12g),Zfp57_extended_-/+_(12r),0.529200,-1,-0.050111,9308,-+,Zfp57-+
14267,chr16:55822169-55822669,Zbtb11,0.030970,0.057198,0.001771,0.001771,Zfp57,True,Zfp57_extended_-/+,Zfp57_extended_-/+_(12g),Zfp57_extended_-/+_(12r),0.664658,-1,-0.116285,7008,-+,Zfp57-+


In [7]:
df_reg = df_scenic.groupby(['TF', 'signs', 'Consensus_name',]).first()[['Region_signature_name', 'Gene_signature_name', ]].sort_values('TF')

scenic_regions = np.sort(df_scenic['Region'].unique()) # .shape
scenic_genes = np.sort(df_scenic['Gene'].unique()) # .shape
scenic_tfs = np.sort(df_scenic['TF'].unique())

num_reg    = len(df_reg)
num_tf     = len(scenic_tfs)
num_gene = len(scenic_genes)
num_region = len(scenic_regions)
print(num_reg, num_tf, num_gene, num_region)
df_reg

38 32 2415 7347


,,,Region_signature_name,Gene_signature_name
TF,signs,Consensus_name,,
Arnt2,++,Arnt2++,Arnt2_extended_+/+_(16r),Arnt2_extended_+/+_(16g)
Atf2,++,Atf2++,Atf2_extended_+/+_(46r),Atf2_extended_+/+_(44g)
Bach2,-+,Bach2-+,Bach2_extended_-/+_(813r),Bach2_extended_-/+_(285g)
Drap1,++,Drap1++,Drap1_extended_+/+_(19r),Drap1_extended_+/+_(13g)
Egr1,++,Egr1++,Egr1_extended_+/+_(1630r),Egr1_extended_+/+_(392g)
Erg,++,Erg++,Erg_extended_+/+_(46r),Erg_extended_+/+_(30g)
Etv5,++,Etv5++,Etv5_extended_+/+_(510r),Etv5_extended_+/+_(284g)
Etv6,++,Etv6++,Etv6_extended_+/+_(39r),Etv6_extended_+/+_(19g)
Fosb,++,Fosb++,Fosb_extended_+/+_(102r),Fosb_extended_+/+_(46g)
